<a href="https://colab.research.google.com/github/raghad612/sic-project-restockai/blob/fatima%2Ft05-eda-sales/notebooks/01_eda_sales_fatima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# T5 EDA of sales (sales_sample_v1) — Issue #21
Owners: Fatima (notebook) and Zahraa. Reviewer: Raghad.
Explore and report only: nothing in this notebook deletes, fills or changes data.
Findings use the training part only (up to 2017-04-30). The last 3 months (test period) are only sanity-checked.

## 0. Setup (run this first)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

REPO = '/content/sic-project-restockai'
!rm -rf {REPO}
!git clone -q -b fatima/t05-eda-sales https://github.com/raghad612/sic-project-restockai.git {REPO}
%cd {REPO}

import sys
sys.path.insert(0, REPO)
from src.config import SALES_SAMPLE   # never type a Drive path by hand

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams['figure.figsize'] = (11, 4)

df = pd.read_parquet(SALES_SAMPLE)
df['date'] = pd.to_datetime(df['date'])
print(df.shape, df['date'].min().date(), 'to', df['date'].max().date())
# expected: (1741225, 8) 2015-08-01 to 2017-07-31

In [ ]:
TRAIN_END = '2017-04-30'
train = df[df['date'] <= TRAIN_END]
print('train rows:', len(train), '| test-period rows (sanity only):', len(df) - len(train))
print('stores:', df['store_nbr'].nunique(), '| items:', df['item_nbr'].nunique(), '| families:', df['family'].nunique())
df.head()

## 1. Negative unit_sales (checkpoint number 1, no change to data)

In [ ]:
neg = df[df['unit_sales'] < 0]
print('negative rows:', len(neg), '| sum:', round(neg['unit_sales'].sum(), 2), '| min:', neg['unit_sales'].min())
pd.crosstab(neg['family'], neg['store_nbr'], margins=True, margins_name='Total')

## 2. Missing store-item-days (checkpoint number 2, reported, not filled)

In [ ]:
n_days = df['date'].nunique()
n_full = df['store_nbr'].nunique() * df['item_nbr'].nunique() * n_days
n_rows = df[['date', 'store_nbr', 'item_nbr']].drop_duplicates().shape[0]
print('days:', n_days, '| possible store-item-days:', n_full, '| with a row:', n_rows)
print(f'share with no row: {(1 - n_rows / n_full) * 100:.1f}%')   # T4 said about 20.6%
print('explicit zero rows:', (df['unit_sales'] == 0).sum())
print('duplicate store-item-day rows:', len(df) - n_rows)

## 3. Kg versus counts (checkpoint number 3)

In [ ]:
# Kg versus counts: share of positive rows where unit_sales is not a whole number (Issue code, full sample)
pos = df[df.unit_sales > 0]
kg_table = (pos.unit_sales % 1 != 0).groupby(pos.family).mean().mul(100).round(1).sort_values(ascending=False)
kg_table.rename('% non-whole rows').to_frame()
# A family with 0% could still be sold by weight: first check only. If this errors, print(df.dtypes) and tell Raghad.

In [ ]:
# items per family (cap = 62)
df.groupby('family')['item_nbr'].nunique().sort_values(ascending=False)

## 4. Extra check: missing share by store

In [ ]:
# missing share by store (does one store have many more gaps?)
per_store = df.groupby('store_nbr').apply(lambda g: 1 - g[['date','item_nbr']].drop_duplicates().shape[0] / (df['item_nbr'].nunique() * n_days))
print((per_store * 100).round(1).sort_values(ascending=False))
# days with no rows at all in a store (e.g. closures)
print('dates in calendar with no sales at all:', len(pd.date_range(df.date.min(), df.date.max())) - n_days)

## 5. Charts

In [ ]:
# a. total units per day
daily = df.groupby('date')['unit_sales'].sum()
fig, ax = plt.subplots(); ax.plot(daily.index, daily.values, lw=0.7)
ax.axvline(pd.Timestamp(TRAIN_END), color='grey', ls='--', label='train end')
ax.set(title='a. Total units sold per day (all 12 stores)', xlabel='Date', ylabel='Units (counts + kg)')
ax.legend(); plt.show()

In [ ]:
# b. total units per week (7-day blocks from 2015-08-01, the team's week definition)
start = df['date'].min()
week_idx = (df['date'] - start).dt.days // 7
weekly = df.groupby(week_idx)['unit_sales'].sum()
weekly = weekly.iloc[:-1] if (df['date'].max() - start).days % 7 != 6 else weekly  # drop partial last week
ax = weekly.plot(marker='.')
ax.set(title='b. Total units per week', xlabel='Week number (from 2015-08-01)', ylabel='Units'); plt.show()

In [ ]:
# c. average units per weekday (average of daily totals, train part)
daily_tr = train.groupby('date')['unit_sales'].sum()
order = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
by_wd = daily_tr.groupby(daily_tr.index.day_name()).mean().reindex(order)
ax = by_wd.plot(kind='bar')
ax.set(title='c. Average daily units by weekday (train part)', xlabel='Weekday', ylabel='Avg units per day'); plt.show()
print((by_wd / by_wd.mean()).round(3))

In [ ]:
# d. total units per family
fam = train.groupby('family')['unit_sales'].sum().sort_values()
ax = fam.plot(kind='barh', figsize=(9, 6))
ax.set(title='d. Total units per family (train part)', xlabel='Units', ylabel='Family'); plt.show()
print((fam / fam.sum() * 100).round(1).sort_values(ascending=False))

In [ ]:
# e. total units per store
st = train.groupby('store_nbr')['unit_sales'].sum().sort_values(ascending=False)
ax = st.plot(kind='bar')
ax.set(title='e. Total units per store (train part)', xlabel='Store number', ylabel='Units'); plt.show()
print('top/bottom store ratio:', round(st.iloc[0] / st.iloc[-1], 2))

In [ ]:
# f. distribution of unit_sales (positive values only, log scale)
pos_tr = train.loc[train['unit_sales'] > 0, 'unit_sales']
bins = np.logspace(np.log10(pos_tr.min()), np.log10(pos_tr.max()), 60)
ax = pos_tr.plot(kind='hist', bins=bins, logx=True)
ax.set(title='f. Distribution of positive unit_sales (log x-axis)', xlabel='unit_sales per row (log)', ylabel='Number of rows'); plt.show()
print(pos_tr.describe(percentiles=[.5, .9, .99, .999]).round(2))

In [ ]:
# g. daily total, March to May 2016, with Apr 16 marked (earthquake)
win = daily['2016-03-01':'2016-05-31']
fig, ax = plt.subplots(); ax.plot(win.index, win.values, marker='.')
ax.axvline(pd.Timestamp('2016-04-16'), color='red', ls='--', label='Apr 16 2016')
ax.set(title='g. Daily total units, Mar-May 2016', xlabel='Date', ylabel='Units'); ax.legend(); plt.show()
a1 = daily['2016-04-01':'2016-04-15'].mean(); a2 = daily['2016-04-16':'2016-04-30'].mean()
print(f'Apr 16-30 vs Apr 1-15 units per day: {(a2/a1 - 1)*100:+.1f}%')   # T4 said +19.3%

## 6. Numbers for the findings
Copy these into the PR description.

In [ ]:
yoy = daily_tr['2016-08-01':'2017-04-30'].mean() / daily_tr['2015-08-01':'2016-04-30'].mean() - 1
summary = {
    'busiest weekday': by_wd.idxmax(), 'busiest vs quietest weekday': round(by_wd.max() / by_wd.min(), 2),
    'top family share %': round(fam.iloc[-1] / fam.sum() * 100, 1), 'top family': fam.index[-1],
    'top/bottom store ratio': round(st.iloc[0] / st.iloc[-1], 2),
    'median positive unit_sales': round(pos_tr.median(), 2), '99.9th pct unit_sales': round(pos_tr.quantile(.999), 1),
    'negative rows': len(neg), 'missing store-item-day %': round((1 - n_rows / n_full) * 100, 1),
    'non-whole positive rows %': round((pos.unit_sales % 1 != 0).mean() * 100, 1),
    'Apr16-30 vs Apr1-15 %': round((a2 / a1 - 1) * 100, 1),
    'growth Aug16-Apr17 vs Aug15-Apr16 %': round(yoy * 100, 1),
}
pd.Series(summary)

## **Findings**
   1. Sunday is the busiest weekday (27.6% above the weekly average) and Thursday the quietest (20.4% below); Sunday averages 1.6x Thursday's daily units (train part). Weekends (Sat + Sun) are clearly higher.
2. BEVERAGES is the top family with 30.2% of all units (train part).
3. The biggest store sells 3.92x the smallest store.
4. Sales are very skewed: median positive unit_sales is 17.0 per row, but the 99.9th percentile is 597.6.
5. 20.2% of store-item-days have no row, and there are 0 explicit zeros: a day with no sale is a missing row. Store 33 has the most gaps (26.0%), store 39 the fewest (15.8%).
6. 18.3% of positive rows are not whole numbers (kg), mostly POULTRY (99.6%), MEATS (98.7%) and PRODUCE (60.3%).
7. After the Apr 16 2016 earthquake, units per day were 19.3% higher (Apr 16-30 vs Apr 1-15), matching T4.
8. Average daily units were 4.2% lower in Aug16-Apr17 than in Aug15-Apr16.

## **Problems for T7/T8 to handle**
- Missing store-item-days (zeros): 20.2% have no row and there are no explicit zeros. T7 decides whether to fill them with 0 (some may be "not stocked yet", not "no sale"). Also 4 calendar dates have no sales in any store (likely closures).
- Negative values: 55 rows (sum -4418.24, min -1806.0), mostly BEVERAGES (27). Probably returns. T7 decides: clip to 0, drop, or keep.
- kg vs counts: POULTRY, MEATS, PRODUCE (and partly PREPARED FOODS, FROZEN FOODS) are sold by weight. Do not mix with count families without care (e.g. in totals or scaling).
- Capped families / DAIRY only 6 items: GROCERY I, BEVERAGES, PRODUCE are at the 62-item cap; DAIRY has 6, LIQUOR,WINE,BEER only 2. Family-level totals reflect the sample, not real store mix.
- April 2016: +19.3% after the earthquake. T8 may need a flag/feature for this period, or T7 may treat it as unusual.
- New: store 33 has 26.0% missing days (highest); sales are heavily skewed (99.9th pct 597.6 vs median 17), so outliers need a rule; min negative value -1806 is one very large return.